# ML · Feature Engineering — Riesgo de Retraso en Entrega

## Propósito

Construir la tabla de features para el modelo de predicción de riesgo de retraso,
usando únicamente información disponible **antes** del despacho del pedido (sin fuga de datos).

## Fuente

- `gld_dataco.supply_chain.desc_ft_orders` — solo registros vigentes (`es_vigente = true`)
- `gld_dataco.supply_chain.desc_dim_fecha` — para extraer mes y día de la semana
- `gld_dataco.supply_chain.desc_dim_shipping_mode` — para el nombre del modo de envío
- `gld_dataco.supply_chain.desc_dim_locations` — para market y región

## Features construidas (sin fuga de datos)

| Feature | Tipo | Descripción |
|---|---|---|
| `shipping_mode` | Categórica | Modo de envío elegido al crear el pedido |
| `days_for_shipment_scheduled` | Numérica | Días prometidos de entrega |
| `market` | Categórica | Mercado geográfico de destino |
| `order_region` | Categórica | Región de destino |
| `es_sospechoso_fraude` | Binaria | Pedido marcado como sospechoso de fraude |
| `mes_pedido` | Numérica | Mes del año (1–12) |
| `dia_semana_pedido` | Numérica | Día de la semana (0=lunes … 6=domingo) |

## Variable objetivo

- `late_delivery_risk` — 1 si el pedido se retrasó, 0 si llegó a tiempo

## Destino

**Tabla:** `gld_dataco.supply_chain.desc_ml_features_orders`

**Grano:** un registro por pedido vigente

## Notas

- Se excluyen pedidos con `es_excluido_otd_leadtime = true` (Shipping canceled),
  ya que no tienen información de riesgo real de entrega.
- Esta tabla se recrea en cada ejecución (overwrite idempotente).

In [0]:
# ============================================================
# ML_FEATURE_ENGINEERING — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog",     "gld_dataco",    "Catálogo Gold")
dbutils.widgets.text("gold_schema",      "supply_chain",  "Esquema Gold")
dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema",  "supply_chain",  "Esquema Metadata")
dbutils.widgets.text("id_ejecucion_manual", "", "Id de ejecución (solo si no corre dentro del Workflow)")

gold_catalog     = dbutils.widgets.get("gold_catalog")
gold_schema      = dbutils.widgets.get("gold_schema")
metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema  = dbutils.widgets.get("metadata_schema")

GOLD_FACT_ORDERS_FQN    = f"{gold_catalog}.{gold_schema}.desc_ft_orders"
GOLD_DIM_FECHA_FQN      = f"{gold_catalog}.{gold_schema}.desc_dim_fecha"
GOLD_DIM_SHIPPING_FQN   = f"{gold_catalog}.{gold_schema}.desc_dim_shipping_mode"
GOLD_DIM_LOCATIONS_FQN  = f"{gold_catalog}.{gold_schema}.desc_dim_locations"
GOLD_ML_FEATURES_FQN    = f"{gold_catalog}.{gold_schema}.desc_ml_features_orders"
TABLA_ETL_LOG           = f"{metadata_catalog}.{metadata_schema}.hd_etl_log"

print(f"Origen principal : {GOLD_FACT_ORDERS_FQN}")
print(f"Destino features : {GOLD_ML_FEATURES_FQN}")

In [0]:
# ============================================================
# ML_FEATURE_ENGINEERING — Celda 2: Inicio de log de ejecución
# ============================================================

import uuid

id_manual = dbutils.widgets.get("id_ejecucion_manual")
try:
    id_ejecucion = dbutils.jobs.taskValues.get(taskKey="prepare_pipeline_control", key="id_ejecucion")
except Exception:
    if not id_manual:
        raise
    id_ejecucion = id_manual

LOG_ID   = str(uuid.uuid4())
NOTEBOOK = "ml_feature_engineering"
CAPA     = "gold"

spark.sql(f"""
INSERT INTO {TABLA_ETL_LOG}
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

try:
    dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)
    print("log_id publicado como Task Value")
except Exception:
    if not id_manual:
        raise
    print("Corrida manual: log_id no se publica como Task Value")

print(f"Registro EN_PROCESO creado: {LOG_ID}")

In [0]:
# ============================================================
# ML_FEATURE_ENGINEERING — Celda 3: Leer Gold
# ============================================================

from pyspark.sql import functions as F

# Solo pedidos vigentes, sin los que tienen Shipping canceled
df_orders = (
    spark.table(GOLD_FACT_ORDERS_FQN)
    .filter(F.col("es_vigente") == True)
    .filter(F.col("es_excluido_otd_leadtime") == False)
)

df_fecha = spark.table(GOLD_DIM_FECHA_FQN).select(
    "fecha_key", "mes", "dia_semana_num"
)

df_shipping = spark.table(GOLD_DIM_SHIPPING_FQN).select(
    "shipping_mode_key", "shipping_mode"
)

df_locations = spark.table(GOLD_DIM_LOCATIONS_FQN).select(
    "location_key", "market", "order_region"
)

print(f"Pedidos vigentes en Gold (base para features): {df_orders.count()}")

In [0]:
# ============================================================
# ML_FEATURE_ENGINEERING — Celda 4: Construir features
# ============================================================

df_features = (
    df_orders.alias("o")
    # Join con dim_fecha para extraer mes y día de la semana del pedido
    .join(df_fecha.alias("f"),
          F.col("o.fecha_pedido_key") == F.col("f.fecha_key"),
          "left")
    # Join con dim_shipping_mode para nombre legible
    .join(df_shipping.alias("sm"),
          F.col("o.shipping_mode_key") == F.col("sm.shipping_mode_key"),
          "left")
    # Join con dim_locations para market y región
    .join(df_locations.alias("l"),
          F.col("o.location_key") == F.col("l.location_key"),
          "left")
    .select(
        # Identificador
        F.col("o.order_id"),

        # --- FEATURES (sin fuga de datos) ---
        F.col("sm.shipping_mode"),
        F.col("o.days_for_shipment_scheduled"),
        F.col("l.market"),
        F.col("l.order_region"),
        F.col("o.es_sospechoso_fraude").cast("int").alias("es_sospechoso_fraude"),
        F.col("f.mes").alias("mes_pedido"),
        F.col("f.dia_semana_num").alias("dia_semana_pedido"),

        # --- VARIABLE OBJETIVO ---
        F.col("o.late_delivery_risk_original").alias("late_delivery_risk"),

        # Metadatos de trazabilidad
        F.col("o.fecha_fin_corte"),
        F.current_timestamp().alias("fecha_features")
    )
)

total      = df_features.count()
nulos_sm   = df_features.filter(F.col("shipping_mode").isNull()).count()
nulos_mkt  = df_features.filter(F.col("market").isNull()).count()
nulos_obj  = df_features.filter(F.col("late_delivery_risk").isNull()).count()

print(f"Total de registros de features : {total}")
print(f"Sin shipping_mode              : {nulos_sm}")
print(f"Sin market                     : {nulos_mkt}")
print(f"Sin variable objetivo          : {nulos_obj}")

In [0]:
# ============================================================
# ML_FEATURE_ENGINEERING — Celda 5: Balance de clases
# (informativo — ayuda a entender si el dataset está desbalanceado)
# ============================================================

print("Distribución de la variable objetivo (late_delivery_risk):")
df_features.groupBy("late_delivery_risk").count().orderBy("late_delivery_risk").show()

In [0]:
# ============================================================
# ML_FEATURE_ENGINEERING — Celda 6: Guardar tabla de features
# (overwrite idempotente — se recrea en cada ejecución)
# ============================================================

(
    df_features
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ML_FEATURES_FQN)
)

print(f"Tabla de features guardada: {GOLD_ML_FEATURES_FQN}")
print(f"Registros escritos: {total}")

In [0]:
# ============================================================
# ML_FEATURE_ENGINEERING — Celda 7: Cerrar log de ejecución
# ============================================================

TABLA_DESTINO    = GOLD_ML_FEATURES_FQN
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE {TABLA_ETL_LOG}
SET estado           = 'EXITOSO',
    fecha_fin        = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")